#### THIRD ATTEMPT

## Data Exploration

### OpTc Dataset
Overview
The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: ([Open Data Repossitory](https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC))

**Project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.

**Chapter goal:**

The goal of this chapter is to load the Parquet files generated and the event-level labelled telemetery made available at ([OpTC event-level labels — GitHub Gist](https://gist.github.com/Hadieeyyy/6cfbb2e2e747e376a2d28df41776b747#file-labels-csv)). After that, I mathched the ids of the malicious events to the corresponding events in the processed data and labelled them as 0/1 where they are benign or malicious, respectively.The resulting dataset exhibited substantial class imbalance, with approximately one malicious event for every 612 benign events. This imbalance is consistent with the rare-event nature of cyberattack detection in operational environments. For example, ([Yang et al. (2024)](https://www.usenix.org/conference/usenixsecurity24/presentation/yang-limin)), analysing 115 million alerts from a real-world SOC, found that only 0.01% of alerts were associated with true attacks or compromises.After that, I flattened the properties column then assigned the labels to the events in the dataset.


In [1]:
# Imports
import json
import pandas as pd
from pathlib import Path



#### 1. Load all processed parquet files into a dataframe

In [4]:
file_path = Path("../../../datasets/OpTC_data/processed")

parquet_files = list(file_path.glob("*.parquet"))


data = pd.concat(
    [pd.read_parquet(file) for file in parquet_files],
    ignore_index=True
)

print(data.shape)

(59405183, 12)


#### 2. Get dataset labels

In [6]:
labels = pd.read_csv("../../../datasets/OpTC_data/labels/labels.csv")
print(labels.head())
labels.info()

                     hostname                                    id  \
0  SysClient0201.systemia.com  43fb9623-3cd1-45ec-ab22-dbe46e75240e   
1  SysClient0201.systemia.com  78fccbc8-58d1-4598-ae21-99f57ee57ed8   
2  SysClient0201.systemia.com  fb89e8be-47a1-418c-9bb8-a4c702694c74   
3  SysClient0201.systemia.com  05e5bde3-8db5-410a-ad75-de676bd14ebb   
4  SysClient0201.systemia.com  1bdb7482-a548-40f2-b648-ea258e6c2448   

                               objectID                               actorID  \
0  96913629-c1c9-4503-9586-4a91de0e7311  af6b49d5-f648-41a4-946d-d92b174bae47   
1  b53c1986-842c-493a-910c-78b55da2575f  96913629-c1c9-4503-9586-4a91de0e7311   
2  b53c1986-842c-493a-910c-78b55da2575f  96913629-c1c9-4503-9586-4a91de0e7311   
3  b53c1986-842c-493a-910c-78b55da2575f  96913629-c1c9-4503-9586-4a91de0e7311   
4  b53c1986-842c-493a-910c-78b55da2575f  96913629-c1c9-4503-9586-4a91de0e7311   

                       timestamp   object   action  
0  2019-09-23T11:23:55.857-04:00 

### 3. Ensure ID datatypes are strings

In [ ]:
data["id"] = data["id"].astype(str)
labels["id"] = labels["id"].astype(str)

# Label telemetry events based on what is in the labels table
data["label"] = data["id"].isin(labels["id"]).astype(int)
data["label"].value_counts()

# From the results, we notice a severe imbalance with about 612:1 benign to malicious cases.

label
0    59308239
1       96944
Name: count, dtype: int64

In [ ]:
### 4. Further Checks

In [ ]:
# Check the share of malicious activity present in downloadded telemetry
matched_labels = labels[labels["id"].isin(data["id"])]

print("Total malicious events in labels.csv:", len(labels))
print("Malicious events found in our telemetry:", len(matched_labels))
print("Telemetry rows labelled malicious:", data["label"].sum())

# 33.2% of malicious events in labels.csv are present in the selected telemetry
# because only selected attack-host groups were downloaded for data manageability

Total malicious events in labels.csv: 292367
Malicious events found in our telemetry: 97044
Telemetry rows labelled malicious: 96944


In [ ]:
# Confirms that the event-level labels are matching actual telemetry rows
malicious_df = data[data["label"] == 1].copy()

malicious_df[
    ["timestamp", "hostname", "id", "object", "action"]
].head(5)

,timestamp,hostname,id,object,action
21118027,2019-09-24 10:39:23.033000-04:00,SysClient0811.systemia.com,a2bd41b8-2f06-4548-8686-e7e53f9db19f,PROCESS,CREATE
21118034,2019-09-24 10:39:23.120000-04:00,SysClient0811.systemia.com,6d71f278-7066-4634-8f45-60ec20457395,MODULE,LOAD
21118035,2019-09-24 10:39:23.120000-04:00,SysClient0811.systemia.com,0c265166-9624-4aa9-9de6-cd50e0677719,MODULE,LOAD
21118036,2019-09-24 10:39:23.120000-04:00,SysClient0811.systemia.com,bb3bc09c-6fc9-44ef-8ed6-e1dd6fc6ea8b,MODULE,LOAD
21118037,2019-09-24 10:39:23.120000-04:00,SysClient0811.systemia.com,1943f7b0-2f99-4c4b-b1ba-a49d65114c51,FILE,READ


In [ ]:
# Ddistribution Checks
print("Malicious events by hostname:")
print(malicious_df["hostname"].value_counts())

print("\nMalicious events by date:")
print(
    pd.to_datetime(malicious_df["timestamp"], utc=True)
      .dt.date
      .value_counts()
      .sort_index()
)

print("\nMalicious events by object:")
print(malicious_df["object"].value_counts())

print("\nMalicious events by action:")
print(malicious_df["action"].value_counts())

# Malicious events occur only on the five selected attack hosts and corresponding attack days.
# They span multiple event types, supporting the validity of the event-level label matching.

Malicious events by hostname:
hostname
SysClient0501.systemia.com    27580
SysClient0201.systemia.com    26681
SysClient0351.systemia.com    18889
SysClient0811.systemia.com    18649
SysClient0051.systemia.com     5145
Name: count, dtype: int64

Malicious events by date:
timestamp
2019-09-23    26681
2019-09-24    46229
2019-09-25    24034
Name: count, dtype: int64

Malicious events by object:
object
SHELL       41996
FLOW        20882
FILE        17837
THREAD       7560
PROCESS      6428
MODULE       2133
REGISTRY      108
Name: count, dtype: int64

Malicious events by action:
action
COMMAND          41996
MESSAGE          15977
CREATE            6792
READ              6262
OPEN              6058
WRITE             5003
START             4903
TERMINATE         4800
LOAD              2133
RENAME            1846
MODIFY             515
REMOTE_CREATE      331
DELETE             220
EDIT                60
ADD                 31
REMOVE              17
Name: count, dtype: int64


#### 5. Flatten propertiess and add labels

In [13]:

processed_files = Path("../../../datasets/OpTC_data/processed")
ready = processed_files / "ready"

ready.mkdir(parents=True, exist_ok=True)

# Get malicious IDs
labels = pd.read_csv("../../../datasets/OpTC_data/labels/labels.csv")
malicious_ids = set(labels["id"].astype(str))

# Get parquet files
parquet_files = list(processed_files.glob("*.parquet"))

print(f"Found {len(parquet_files)} files")


# Flatten each file separately
for i, file in enumerate(parquet_files, start=1):

    print(f"\n[{i}/{len(parquet_files)}] Processing {file.name}")

    df = pd.read_parquet(file)

    # Add labels
    df["id"] = df["id"].astype(str)
    df["label"] = df["id"].isin(malicious_ids).astype(int)

    # Expand properties
    properties_df = pd.json_normalize(df["properties"])
    properties_df.index = df.index

    df = df.drop(columns=["properties"])
    df = pd.concat([df, properties_df], axis=1)

    # Save processed file
    output_path = ready / file.name
    df.to_parquet(output_path, index=False)

    print(
        f"Saved {file.name} | "
        f"{len(df):,} rows | "
        f"{df.shape[1]} columns | "
        f"{df['label'].sum():,} malicious"
    )

    del df, properties_df


print("\nDone.")

Found 41 files

[1/41] Processing 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0060.parquet
Saved 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0060.parquet | 143,207 rows | 62 columns | 0 malicious

[2/41] Processing 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0070.parquet
Saved 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0070.parquet | 139,443 rows | 62 columns | 0 malicious

[3/41] Processing 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0056.parquet
Saved 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0056.parquet | 138,672 rows | 62 columns | 0 malicious

[4/41] Processing 2019-09-23_AIA-201-225_sysclient0204.parquet
Saved 2019-09-23_AIA-201-225_sysclient0204.parquet | 4,049,225 rows | 59 columns | 0 malicious

[5/41] Processing 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0069.parquet
Saved 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0069.parquet | 142,633 rows | 62 columns | 0 malicious

[6/41

In [ ]:
# flattened_files = Path(
#     "../../../datasets/OpTC_data/processed/ready"
# )

# files = list(flattened_files.glob("*.parquet"))

# print(f"Number of files: {len(files)}")

# sample = pd.read_parquet(files[0])

# print(sample.shape)
# print(sample.columns.tolist())

# sample.head()
# print("properties" in sample.columns)
# print("label" in sample.columns)

# print(sample["label"].value_counts())

Number of files: 41
(143207, 62)
['action', 'actorID', 'hostname', 'id', 'object', 'objectID', 'pid', 'ppid', 'principal', 'tid', 'timestamp', 'label', 'acuity_level', 'base_address', 'command_line', 'context_info', 'data', 'dest_ip', 'dest_port', 'direction', 'end_time', 'file_path', 'image_path', 'info_class', 'key', 'l4protocol', 'logon_id', 'module_path', 'name', 'new_path', 'parent_image_path', 'path', 'payload', 'privileges', 'requesting_domain', 'requesting_logon_id', 'requesting_user', 'service_type', 'sid', 'size', 'src_ip', 'src_pid', 'src_port', 'src_tid', 'stack_base', 'stack_limit', 'start_address', 'start_time', 'start_type', 'subprocess_tag', 'task_name', 'task_pid', 'task_process_uuid', 'tgt_pid', 'tgt_pid_uuid', 'tgt_tid', 'type', 'user', 'user_name', 'user_stack_base', 'user_stack_limit', 'value']
False
True
label
0    143207
Name: count, dtype: int64


In [ ]:
# total_rows = 0
# total_malicious = 0

# for file in files:
#     df = pd.read_parquet(file, columns=["label"])

#     total_rows += len(df)
#     total_malicious += df["label"].sum()

# print(f"Total rows: {total_rows:,}")
# print(f"Malicious: {total_malicious:,}")
# print(f"Benign/unlabelled: {total_rows - total_malicious:,}")
# print(f"Malicious percentage: {total_malicious / total_rows * 100:.3f}%")

Total rows: 59,405,183
Malicious: 96,944
Benign/unlabelled: 59,308,239
Malicious percentage: 0.163%
